In [1]:
import numpy as np
import pandas as pd

In [ ]:
raw_data = pd.read_csv('fbref_big5_players_2019_2025_all_stat_types.csv')

C:\Users\User\AppData\Local\Temp\ipykernel_22128\3549644905.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv('fbref_big5_players_2019_2025_all_stat_types.csv')


In [28]:
# create dictionary of stat_type -> list of non-empty columns
# for each stat_type we look at rows of that type and record any column
# where at least one value is not NaN.
df = raw_data.copy()
stat_columns = {}
if 'stat_type' in df.columns:
    for st in df['stat_type'].dropna().unique():
        subset = df[df['stat_type'] == st]
        # consider all columns except stat_type itself (and maybe player/season)
        nonempty = [col for col in subset.columns if subset[col].notna().any()]
        # remove the stat_type column from list since it's always non-null
        if 'stat_type' in nonempty:
            nonempty.remove('stat_type')
        stat_columns[st] = nonempty
else:
    print('no stat_type column available to inspect')
    


# also check per stat_type if you want to ensure coverage inside each group
columns_all_na_by_type = {}
if 'stat_type' in df.columns:
    for st in df['stat_type'].dropna().unique():
        subset = df[df['stat_type']==st]
        cols = subset.columns[subset.isna().all()].tolist()
        columns_all_na_by_type[st] = cols


# split the dataframe by stat_type using the previously computed column lists,
# then merge everything back together so we end up with one row per player+season
# without adding any suffixes; when columns conflict we'll keep the first non-null value.

# start with a base of unique player/season combinations
merged = df[['Player','Season_End_Year']].drop_duplicates().copy()

for st, cols in stat_columns.items():
    # ensure Player and Season are included for the join and remove duplicates
    use_cols = ['Player', 'Season_End_Year'] + [c for c in cols if c not in ('Player','Season')]
    # make sure the column list has unique names (avoid duplicates in `cols` itself)
    use_cols = list(dict.fromkeys(use_cols))

    subset = df[df['stat_type'] == st][use_cols].copy()
    subset = subset.drop_duplicates(['Player','Season_End_Year'])
    # merge with suffix pattern: left keeps original, right duplicates get _dup
    merged = pd.merge(merged, subset, on=['Player','Season_End_Year'], how='outer', suffixes=('','_dup'))
    # drop any duplicate columns created by the merge (those ending in _dup)
    dup_cols = [c for c in merged.columns if c.endswith('_dup')]
    if dup_cols:
        # for each duplicate, fill missing values from the _dup column then drop it
        for c in dup_cols:
            orig = c[:-4]
            merged[orig] = merged[orig].combine_first(merged[c])
        merged = merged.drop(columns=dup_cols)

print('merged shape:', merged.shape)

# Start from full merged dataframe
df = merged.copy()

# Clean spacing just in case
df["Pos"] = df["Pos"].astype(str).str.replace(" ", "", regex=False)

# Deterministic mapping exactly as discussed
pos_mapping = {
    "DF,MF": "DEF_MF",
    "MF,DF": "DEF_MF",
    "FW,MF": "ATT_MF",
    "MF,FW": "ATT_MF",
    "DF,FW": "DF",
    "FW,DF": "DF",
}

# Update positions (single positions remain unchanged)
df["Pos_updated"] = df["Pos"].replace(pos_mapping)

def modeling_group(pos):
    if pos == "GK":
        return "GK"
    if pos == "DF":
        return "DEF"
    if pos == "FW":
        return "FWD"
    if pos in ["MF", "ATT_MF", "DEF_MF"]:
        return "MID"
    return None

df["position_group_model"] = df["Pos_updated"].apply(modeling_group)

df['Age'] = df['Age'].astype(str).str[:2].replace('na', None).astype('Int64')


merged shape: (19754, 210)


In [38]:
pd.DataFrame({"column_name": df.columns}).to_csv("all_columns.csv", index=False)

In [36]:
pd.set_option('display.max_columns', None)

In [30]:
numeric_cols

Index(['Season_End_Year', 'Age', 'Born', 'MP_Playing', 'Starts_Playing',
       'Min_Playing', 'Mins_Per_90_Playing', 'Gls', 'Ast', 'G+A',
       ...
       'Att_Goal', 'Launch_percent_Goal', 'AvgLen_Goal', 'Opp_Crosses',
       'Stp_Crosses', 'Stp_percent_Crosses', '#OPA_Sweeper',
       '#OPA_per_90_Sweeper', 'AvgDist_Sweeper', 'Att (GK)_Passes'],
      dtype='object', length=204)

In [13]:
# convert column names to lowercase
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

# split the dataframe into separate dataframes for each position group
df_gk = df[df["position_group_model"] == "GK"].copy()
df_def = df[df["position_group_model"] == "DEF"].copy()
df_mid = df[df["position_group_model"] == "MID"].copy()
df_fwd = df[df["position_group_model"] == "FWD"].copy()

In [14]:
print("GK:", df_gk.shape)
print("DEF:", df_def.shape)
print("MID:", df_mid.shape)
print("FWD:", df_fwd.shape)

GK: (2282, 212)
DEF: (6294, 212)
MID: (8573, 212)
FWD: (2600, 212)
